In [ ]:
import os
from sentence_transformers import SentenceTransformer, CrossEncoder

############
# import sys
# sys.path.append('/content/drive/MyDrive/Nakamo')
# the above 2 lines were used to locate the code using G.Drive
############

# Import our custom modules
from data_processing.parser import parse_pdfs_in_folder
from data_processing.chunker import chunk_data
from vector_store.db_manager import clear_vector_store, create_and_populate_vector_store
from llm.local_llm import LocalLLM
from rag_agents.rag_pipelines import (
    get_rag_response
)


### Configurations

In [ ]:
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "med_device_regulations"
pdf_folder = 'documents'

# --- Retrieval models Configuration ---
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
# --- LLM Configuration ---
LLM_MODEL_ID = "Qwen/Qwen3-0.6B-GGUF"
LLM_MODEL_FILE = "Qwen3-0.6B-Q8_0.gguf"



#### Load the models

In [ ]:

# load the models
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

reranker = CrossEncoder(RERANKER_MODEL)

local_llm = LocalLLM(LLM_MODEL_ID, LLM_MODEL_FILE, verbose=False)


## Document Processing and DB populating

#### Parse the pdf files

In [4]:
pages_data = parse_pdfs_in_folder(pdf_folder)

Starting structured parsing for 3 PDF file(s)...
Processing 'FDA_Design_Control_Guidance.pdf' with avg font size 12.17...
Processing 'FDA_Policy_Device_Software_Functions.pdf' with avg font size 12.14...
Processing 'WHO_Medical_Device_Regulations.pdf' with avg font size 11.07...
Structured parsing complete. Extracted 1384 blocks.

--- Sample of Structured Data ---
Type: paragraph
Source: FDA_Design_Control_Guidance.pdf, Page: 7
Text: FOREWORD................................................................................................................... i
PREFACE....................

Type: paragraph
Source: FDA_Design_Control_Guidance.pdf, Page: 9
Text: Introduction
               3/11/97
Page 1...

Type: paragraph
Source: FDA_Design_Control_Guidance.pdf, Page: 9
Text: This guidance is intended to assist manufacturers in understanding quality system
requirements concerning design controls. Assistance is provided by i...

Type: paragraph
Source: FDA_Design_Control_Guidance.pdf, Page: 

#### Chunk the parsed data

In [6]:
if pages_data:
    chunked_data = chunk_data(pages_data, chunk_size = 1024, chunk_overlap = 256)
    # Print a sample of the output to verify
    if chunked_data:
        print("\n--- Sample of Chunked Data ---")
        sample_chunk = chunked_data[10] # Picking a random chunk
        print(f"Source: {sample_chunk['metadata']['source']}")
        print(f"Page: {sample_chunk['metadata']['page']}")
        print(f"Chunk Text: '{sample_chunk['text']}'")
        print("---------------------------------")


Starting contextual chunking on 1384 blocks...
Contextual chunking complete. Created 1240 chunks.

--- Sample of Chunked Data ---
Source: FDA_Design_Control_Guidance.pdf
Page: 5
Chunk Text: 'This Guidance relates to
FDA 21 CFR 820.30 and Sub-clause 4.4 of ISO 9001

The response of medical device manufacturers and other interested parties to the March, 1996 draft
version of this guidance has significantly influenced this latest version.  Most comments centered
on the complaint that the guidance was too prescriptive.  Therefore, it has been rewritten to be
more pragmatic, focusing on principles rather than specific practices.'
---------------------------------


In [ ]:
# clear_vector_store(CHROMA_PATH, COLLECTION_NAME)

Initializing ChromaDB client to clear the store...
Attempting to delete collection: 'med_device_regulations'
Vector store collection has been successfully cleared.


#### Populate the DB

In [7]:
if chunked_data:
    create_and_populate_vector_store(chunked_data, CHROMA_PATH, COLLECTION_NAME, embedding_model)

Initializing ChromaDB client...
Getting or creating ChromaDB collection: 'med_device_regulations'
Generating embeddings for 1240 documents...


Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Adding documents to the collection in batches...
Added batch 1/13
Added batch 2/13
Added batch 3/13
Added batch 4/13
Added batch 5/13
Added batch 6/13
Added batch 7/13
Added batch 8/13
Added batch 9/13
Added batch 10/13
Added batch 11/13
Added batch 12/13
Added batch 13/13
Vector store creation and population complete.
Total documents in collection: 1240


## Retrieve information based on User Queries.

In [5]:
user_query = "Hello"

get_rag_response(
    user_query=user_query,
    local_llm=local_llm,
    embedding_model=embedding_model,
    reranker=reranker,
    CHROMA_PATH=CHROMA_PATH,
    COLLECTION_NAME=COLLECTION_NAME,
    final_n_results=5,
    initial_fetch=15,
    max_new_tokens=512,
    rag_temp=0.05,
    direct_temp=0.3
)


--- 1. Orchestrator Agent: Routing Query ---
{"action": "ANSWER_DIRECTLY"} 
Orchestrator Decision: action='ANSWER_DIRECTLY'
--- Action: ANSWER_DIRECTLY ---
{
  "id": "chatcmpl-ec87ad70-9cb4-4b1b-9c52-3e59455cadc5",
  "object": "chat.completion",
  "created": 1755900262,
  "model": "/root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-GGUF/snapshots/23749fefcc72300e3a2ad315e1317431b06b590a/Qwen3-0.6B-Q8_0.gguf",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "<think>\n\n</think>\n\nHello! How can I assist you today?"
      },
      "logprobs": null,
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 28,
    "completion_tokens": 13,
    "total_tokens": 41
  }
}

✅ Final Answer


<think>

</think>

Hello! How can I assist you today?

In [4]:
user_query = "What are the design control requirements for verification and validation?"

get_rag_response(
    user_query=user_query,
    local_llm=local_llm,
    embedding_model=embedding_model,
    reranker=reranker,
    CHROMA_PATH=CHROMA_PATH,
    COLLECTION_NAME=COLLECTION_NAME,
    final_n_results=5,
    initial_fetch=15,
    max_new_tokens=512,
    rag_temp=0.05,
    direct_temp=0.3
)


--- 1. Orchestrator Agent: Routing Query ---
{"action": "SEARCH"} 
Orchestrator Decision: action='SEARCH'
--- Action: SEARCH ---

--- 2 Query Decomposition Agent: Generating Subqueries ---
Generated 4 queries: [
  "What are the design control requirements for verification and validation?",
  "Design Control Requirements: Verification & Validation",
  "Verification Design Controls in Software Engineering, What is Required?",
  "Requirements Analysis of Verification and Validation"
]

--- 3. Retrieval Agent: Retrieving Context ---

--- Retrieving docs for 4 queries ---
  - Querying: 'What are the design control requirements for verification and validation?'
  - Querying: 'Design Control Requirements: Verification & Validation'
  - Querying: 'Verification Design Controls in Software Engineering, What is Required?'
  - Querying: 'Requirements Analysis of Verification and Validation'
Retrieved 33 unique documents in total.

--- Pre-selecting top 15 docs from 33 unique docs ---

Reranking th

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Context retrieved successfully.
--------------------------------------------------
[Source: FDA_Design_Control_Guidance.pdf, Page: 42] §820.3(aa) Verification means confirmation by examination and provision of objective
evidence that specified requirements have been fulfilled.

Whereas verification is a detailed examination of aspects of a design at various stages in
the development, design validation is a cumulative summation of all efforts to assure that
the design will conform with user needs and intended use(s), given expected variations in
components, materials, manufacturing processes, and the use environment.

---

[Source: FDA_Design_Control_Guidance.pdf, Page: 49] APPLICATION OF DOCUMENT AND CHANGE CONTROLS TO DESIGN.  The
design control system has to be concerned with the creation and revision of documents, as
well as the management of finished documents.  Additional mechanisms are required to
provide needed flexibility while preserving the integrity of design documentation. 

<think>

</think>

Design control requirements for verification and validation include:

- Verification means confirmation by examination and provision of objective evidence that specified requirements have been fulfilled. [Source: FDA_Design_Control_Guidance.pdf, Page: 42]

- Design verification shall confirm that the design output meets the design input requirements. [Source: FDA_Design_Control_Guidance.pdf, Page: 37]

- Design change procedures must include re-verifying and re-validating the design. [Source: FDA_Design_Control_Guidance.pdf, Page: 39]

- Design changes should be documented and communicated to developers to ensure the total impact of the change is determined. [Source: FDA_Design_Control_Guidance.pdf, Page: 39]

In [6]:
user_query = "Is our AI-powered MRI analysis tool considered a medical device software?"

get_rag_response(
    user_query=user_query,
    local_llm=local_llm,
    embedding_model=embedding_model,
    reranker=reranker,
    CHROMA_PATH=CHROMA_PATH,
    COLLECTION_NAME=COLLECTION_NAME,
    final_n_results=5,
    initial_fetch=15,
    max_new_tokens=512,
    rag_temp=0.05,
    direct_temp=0.3
)


--- 1. Orchestrator Agent: Routing Query ---
{"action": "SEARCH"} 
Orchestrator Decision: action='SEARCH'
--- Action: SEARCH ---

--- 2 Query Decomposition Agent: Generating Subqueries ---
Generated 3 queries: [
  "Is our AI-powered MRI analysis tool considered a medical device software?",
  "Is our AI-powered MRI analysis tool a medical device software?",
  "Are there any limitations in the current state of this technology?"
]

--- 3. Retrieval Agent: Retrieving Context ---

--- Retrieving docs for 3 queries ---
  - Querying: 'Is our AI-powered MRI analysis tool considered a medical device software?'
  - Querying: 'Is our AI-powered MRI analysis tool a medical device software?'
  - Querying: 'Are there any limitations in the current state of this technology?'
Retrieved 36 unique documents in total.

--- Pre-selecting top 15 docs from 36 unique docs ---

Reranking the top 15 unique documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Context retrieved successfully.
--------------------------------------------------
[Source: FDA_Policy_Device_Software_Functions.pdf, Page: 42] Answer: Software used in the production process for medical devices, or for collecting, 
storing and maintaining quality system data collection for medical devices (including 
complaint submissions) is not considered a medical device on its own. This software does not 
meet the definition of medical device, but is part of the quality system. However this 
software is required to comply with the appropriate good manufacturing practices (GMP) 
regulations.94

93 Available at https://www.fda.gov/regulatory-information/search-fda-guidance-documents/electronic-source-
data-clinical-investigations.
94 See 21 CFR Part 820.

---

[Source: FDA_Policy_Device_Software_Functions.pdf, Page: 41] Answer: Software used for data collection in clinical studies (such as electronic Patient 
Reported Outcomes (ePRO) apps) is not considered on its own to be a device

<think>

</think>

[Source: FDA_Policy_Device_Software_Functions.pdf, Page: 42] Software used in the production process for medical devices, or for collecting, storing, and maintaining quality system data collection for medical devices (including complaint submissions) is not considered a medical device on its own. This software does not meet the definition of medical device, but is part of the quality system. However this software is required to comply with the appropriate good manufacturing practices (GMP) regulations.94

[Source: FDA_Policy_Device_Software_Functions.pdf, Page: 41] Software used for data collection in clinical studies (such as electronic Patient Reported Outcomes (ePRO) apps) is not considered on its own to be a device software.

[Source: FDA_Policy_Device_Software_Functions.pdf, Page: 7] FDA has previously clarified that when a software application is used to analyze medical device data, it has traditionally been regulated as an accessory to a medical device or as medical device software. In 2014, the International Medical Device Regulators Forum established globally harmonized vocabulary for such software applications and defined the term “Software as a Medical Device (SaMD).”14

[Source: FDA_Policy_Device_Software_Functions.pdf, Page: 38] FDA recognizes that this guidance does not describe all types of software used in healthcare. Some manufacturers may be unsure whether their software function is considered a medical device that is subject to regulatory oversight, or whether their medical device could be under FDA’s intent to exercise enforcement discretion. If the device is subject to regulatory oversight, manufacturers may have questions about which regulatory requirements are applicable to their software function.

In [7]:
user_query = "What documentation is needed for a mobile app that monitors heart rate?"

get_rag_response(
    user_query=user_query,
    local_llm=local_llm,
    embedding_model=embedding_model,
    reranker=reranker,
    CHROMA_PATH=CHROMA_PATH,
    COLLECTION_NAME=COLLECTION_NAME,
    final_n_results=5,
    initial_fetch=15,
    max_new_tokens=512,
    rag_temp=0.05,
    direct_temp=0.3
)

--- 1. Orchestrator Agent: Routing Query ---
{"action": "SEARCH"} 
Orchestrator Decision: action='SEARCH'
--- Action: SEARCH ---

--- 2 Query Decomposition Agent: Generating Subqueries ---
Generated 3 queries: [
  "What documentation is needed for a mobile app that monitors heart rate?",
  "How to document the features of an application with real-time data monitoring?",
  "Can you provide me with step-by-step instructions on how to create and maintain documentation in this context?"
]

--- 3. Retrieval Agent: Retrieving Context ---

--- Retrieving docs for 3 queries ---
  - Querying: 'What documentation is needed for a mobile app that monitors heart rate?'
  - Querying: 'How to document the features of an application with real-time data monitoring?'
  - Querying: 'Can you provide me with step-by-step instructions on how to create and maintain documentation in this context?'
Retrieved 38 unique documents in total.

--- Pre-selecting top 15 docs from 38 unique docs ---

Reranking the top

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Context retrieved successfully.
--------------------------------------------------
[Source: FDA_Policy_Device_Software_Functions.pdf, Page: 15] into a regulated medical device by using attachments, display screens, or sensors 
or by including functionalities similar to those of currently regulated medical 
devices. Software functions that use attachments, display screens, sensors, or 
other such similar components to transform a mobile platform into a regulated 
medical device are required to comply with the device classification associated 
with the transformed platform.

FDA has cleared several mobile medical apps with attachments to a mobile 
platform. Specifically, patient monitoring mobile apps that monitor a patient 
for heart rate variability from a signal produced by an electrocardiograph, 
vectorcardiograph, or blood pressure monitor are classified as cardiac 
monitoring software under 21 CFR 870.2300 (Cardiac monitor (including

---

[Source: FDA_Policy_Device_Software_Functi

<think>

</think>

To document a mobile app that monitors heart rate, the following documentation is required:

- A **software function** that uses an **electrocardiograph (ECG)** or **blood pressure monitor** to measure heart rate variability (HRV) is required to comply with the device classification under 21 CFR 870.2300 (Cardiac monitor). This is supported by the citation [Source: FDA_Policy_Device_Software_Functions.pdf, Page: 15].

- Additionally, the app must include **software functions** that use **attachments, display screens, or sensors** to transform the mobile platform into a regulated medical device. This includes functions like **heart rate monitoring**, **ECG signal processing**, or **blood pressure monitoring**, as mentioned in [Source: FDA_Policy_Device_Software_Functions.pdf, Page: 15].

- The app must also include documentation that specifies the **software function** and the **device classification** associated with the transformed platform, as required by the FDA. This is supported by the citation [Source: FDA_Policy_Device_Software_Functions.pdf, Page: 15].

In [8]:
user_query = "Compare FDA and WHO approaches to risk management for medical devices"

get_rag_response(
    user_query=user_query,
    local_llm=local_llm,
    embedding_model=embedding_model,
    reranker=reranker,
    CHROMA_PATH=CHROMA_PATH,
    COLLECTION_NAME=COLLECTION_NAME,
    final_n_results=5,
    initial_fetch=15,
    max_new_tokens=512,
    rag_temp=0.05,
    direct_temp=0.3
)

--- 1. Orchestrator Agent: Routing Query ---
{"action": "SEARCH"} 
Orchestrator Decision: action='SEARCH'
--- Action: SEARCH ---

--- 2 Query Decomposition Agent: Generating Subqueries ---
Generated 4 queries: [
  "Compare FDA and WHO approaches to risk management for medical devices",
  "FDA vs WHO: Risk Management Approaches for Medical Devices",
  "How do FDA and WHO differ in their risk management approaches to medical devices?",
  "What are the key differences between FDA and WHO's methods of managing risks in healthcare products?"
]

--- 3. Retrieval Agent: Retrieving Context ---

--- Retrieving docs for 4 queries ---
  - Querying: 'Compare FDA and WHO approaches to risk management for medical devices'
  - Querying: 'FDA vs WHO: Risk Management Approaches for Medical Devices'
  - Querying: 'How do FDA and WHO differ in their risk management approaches to medical devices?'
  - Querying: 'What are the key differences between FDA and WHO's methods of managing risks in healthcare pro

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Context retrieved successfully.
--------------------------------------------------
[Source: FDA_Policy_Device_Software_Functions.pdf, Page: 14] Contains Nonbinding Recommendations

As described in this guidance, FDA intends to apply its regulatory oversight to only those 
software functions that are medical devices and whose functionality could pose a risk to a 
patient’s safety if the device were to not function as intended. This approach to overseeing 
device software functions is consistent with our existing approach to overseeing medical 
device functionality of a product and the risks it poses to patients regardless of the shape, 
size, or the platform. FDA believes that this subset of device software functions poses the 
same or similar potential risks to the public health as currently regulated devices if they fail 
to function as intended.

---

[Source: WHO_Medical_Device_Regulations.pdf, Page: 13] CHAPTER 2
Medical device safety

• It is a risk management issue

---

[Source:

<think>

</think>

[Source: FDA_Policy_Device_Software_Functions.pdf, Page: 14] Contains Nonbinding Recommendations  
[Source: WHO_Medical_Device_Regulations.pdf, Page: 13] CHAPTER 2  
Medical device safety  

[Source: WHO_Medical_Device_Regulations.pdf, Page: 14] 2.1 Medical device safety and risk management  
Safety can only be considered in relative terms. All devices carry a certain degree of risk and could cause problems in specific circumstances. Many medical device problems cannot be detected until extensive market experience is gained. For example, an implantable device may fail in a manner that was not predictable at the time of implantation; the failure may reflect conditions unique to certain patients. For other devices, component failure can also be unpredictable or random. The current approach to device safety is to estimate the potential of a device becoming a hazard that could result in safety problems and harm. This estimate is often referred to as the risk assessment.  

[Source: WHO_Medical_Device_Regulations.pdf, Page: 14] 2.1 Medical device safety and risk management  
Safety can only be considered in relative terms. All devices carry a certain degree of risk and could cause problems in specific circumstances. Many medical device problems cannot be detected until extensive market experience is gained. For example, an implantable device may fail in a manner that was not predictable at the time of implantation; the failure may reflect conditions unique to certain patients. For other devices, component failure can also be unpredictable or random. The current approach to device safety is to estimate the potential of a device becoming a hazard that could result in safety problems and harm. This estimate is often referred to as the risk assessment.